In [ ]:
# 2026-11-11
import pandas as pd
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from django.core.exceptions import ObjectDoesNotExist
from apis_core.apis_metainfo.models import Collection, Uri, TempEntityClass
from apis_core.apis_entities.models import Person
from apis_core.apis_vocabularies.models import ProfessionType


In [ ]:
col = Collection.objects.get(id=122)
domain = "spielplan-oper_1869–1945"

In [ ]:
df = gsheet_to_df("1WxCkQJHbUMzp-nePrlC2V9pu77UiqGoeKphW2gPzlDk")

In [ ]:
df.head(3)

In [ ]:
# adds opern-uris to existing entities
no_match = []
for i, row in tqdm(df.iterrows(), total=len(df)):
    if pd.isna(row["pmb-nummer"]):
        continue
    else:
        pmb_id = int(row["pmb-nummer"])
    try:
        entity = TempEntityClass.objects.get(id=pmb_id)
    except ObjectDoesNotExist:
        try:
            entity = Uri.objects.get(uri=f"https://pmb.acdh.oeaw.ac.at/entity/{row['pmb-nummer']}/").entity
        except ObjectDoesNotExist:
            no_match.append(row["pmb-nummer"])
            continue
    entity.collection.add(col)
    bt_uri, _ = Uri.objects.get_or_create(uri=row["uri"], entity=entity, domain=domain)
for x in no_match:
    print(x)

In [ ]:
# import new persons
df = gsheet_to_df("1H4D6V_pYDvswBBvSbxv79Et4F9LQO1qnCuIGGR-v484")
df.head(3)

In [ ]:
professions = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
    pmb_uri, _ = Uri.objects.get_or_create(uri=row["uri"], domain=domain)
    entity = pmb_uri.entity
    if entity:
        person = Person.objects.get(id=entity.id)
        pass
    else:
        item = {
            "name": row["surname"][:254] if pd.notna(row["surname"]) else "",
            "first_name": row["forename"][:254] if pd.notna(row["forename"]) else "",
        }
        
        entity = Person.objects.create(**item)
        try:
            entity.start_date_written = str(int(row["birth_year"]))
        except:
            pass
        try:
            entity.end_date_written = str(int(row["death_year"]))
        except:
            pass

        entity.save()
        person = Person.objects.get(id=entity.id)
        try:
            job = professions[row["occupations"]]
        except KeyError:
            job = ProfessionType.objects.get_or_create(name=row["occupations"])[0]
            professions[row["occupations"]] = job
        person.profession.add(job)
        pmb_uri.entity = entity
        pmb_uri.save()
    entity.collection.add(col)